# 03 · Particiones C++ con Kubernetes

**Objetivo:** ejecutar el ETL mediante un Indexed Job con dos Pods, recuperar las salidas persistentes y reunirlas con un Job C++ adicional.

La prueba usa un nodo local K3s creado para esta entrega. Los dos Pods trabajan sobre particiones distintas en la misma máquina; no se presenta como una prueba entre máquinas físicas.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "cpp/etl.cpp").exists():
    raise RuntimeError("Abrir y ejecutar el notebook desde la carpeta del proyecto")
sys.path.insert(0, str(ROOT / "scripts"))
OUT = ROOT / "resultados_experimentos"
entorno = json.loads((OUT / "entorno.json").read_text())
mediciones = pd.read_csv(OUT / "mediciones.csv")
resumen = pd.read_csv(OUT / "resumen_rendimiento.csv")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
print("Dataset SHA-256:", entorno["dataset_sha256"])
print("Compilación:", entorno["compilacion"])
print("Kubernetes:", entorno["kubernetes"])


Dataset SHA-256: ca65e96d26e1e7cb58b0b5ec1c642fbcb24926684077cbee5cf2864251ce9728
Compilación: g++ 13.3.0; -std=c++17 -O3 -Wall -Wextra -Wpedantic -fopenmp
Kubernetes: v1.34.13+k3s1


## 1. Clúster y almacenamiento

`scripts/preparar_entorno.py` reutiliza la imagen OpenMP y prepara `etl-k3s`, sin reiniciar los contenedores existentes. El servidor K3s de laboratorio funciona dentro de Docker; no se expone una API pública. Sus recursos se limitan a 4 CPU y 1800 MiB de memoria.

Los manifiestos usan el namespace `etl-cpp` y un PVC de 1 GiB. `ReadWriteOnce` es adecuado para esta prueba de un nodo; una prueba multinodo necesitaría revisar el almacenamiento compartido.

```bash
docker exec etl-k3s kubectl get nodes
docker exec etl-k3s kubectl apply -f /tmp/etl-cpp-manifiestos/almacenamiento.yaml
```


In [2]:
print("Versión probada:", entorno["kubernetes"])
print((ROOT / "k8s/cpp/almacenamiento.yaml").read_text())
print((ROOT / "k8s/cpp/job-particiones.yaml").read_text())

Versión probada: v1.34.13+k3s1
apiVersion: v1
kind: Namespace
metadata:
  name: etl-cpp
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: resultados-cpp
  namespace: etl-cpp
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 1Gi

apiVersion: batch/v1
kind: Job
metadata:
  name: etl-cpp-particiones
  namespace: etl-cpp
  labels:
    app: etl-cpp
spec:
  completions: 2
  parallelism: 2
  completionMode: Indexed
  backoffLimit: 0
  activeDeadlineSeconds: 180
  template:
    spec:
      securityContext:
        fsGroup: 1000
      restartPolicy: Never
      containers:
        - name: etl
          image: etl-distribuido-cpp:local
          imagePullPolicy: Never
          args:
            - --input
            - /datos/COVID19MEXICO.csv
            - --output
            - /resultados/parte_$(INDICE).csv
            - --audit
            - /resultados/parte_$(INDICE).json
            - --mode
            - omp
            - --threads
           

## 2. Ejecución paralela y unión

`parallelism: 2`, `completions: 2` y `completionMode: Indexed` crean dos tareas identificadas por índice. Cada Pod usa dos hilos OpenMP y escribe su propio CSV, auditoría e índices. La unión se lanza después de comprobar que el Job de particiones completó las dos tareas.

```bash
docker exec etl-k3s kubectl apply -f /tmp/etl-cpp-manifiestos/job-particiones.yaml
docker exec etl-k3s kubectl -n etl-cpp wait --for=condition=complete job/etl-cpp-particiones --timeout=180s
docker exec etl-k3s kubectl apply -f /tmp/etl-cpp-manifiestos/job-union.yaml
docker exec etl-k3s kubectl -n etl-cpp wait --for=condition=complete job/etl-cpp-union --timeout=120s
```


In [3]:
carpeta = OUT / "ejecuciones/kubernetes_2x2_05"
jobs = json.loads((carpeta / "jobs.json").read_text())
display(pd.DataFrame([{"job": j["metadata"]["name"], "completados": j["status"].get("succeeded", 0),
                       "esperados": j["spec"].get("completions", 1)} for j in jobs["items"]]))
pods = json.loads((carpeta / "pods.json").read_text())
display(pd.DataFrame([{"pod": p["metadata"]["name"], "fase": p["status"]["phase"],
                       "indice": p["metadata"].get("annotations", {}).get("batch.kubernetes.io/job-completion-index", "lector/unión")}
                      for p in pods["items"]]))
print((carpeta / "ejecucion.log").read_text())

,job,completados,esperados
0,etl-cpp-particiones,2,2
1,etl-cpp-union,1,1


,pod,fase,indice
0,etl-cpp-lector,Running,lector/unión
1,etl-cpp-particiones-0-n7r26,Succeeded,0
2,etl-cpp-particiones-1-c8gd7,Succeeded,1
3,etl-cpp-union-vmlj9,Succeeded,lector/unión


job.batch/etl-cpp-particiones created
job.batch/etl-cpp-particiones condition met
job.batch/etl-cpp-union created
job.batch/etl-cpp-union condition met



## 3. Recuperación y validación

El Pod lector monta el mismo PVC y permite leer los archivos con la utilidad mínima `etl --read-file`. La imagen no necesita shell ni `tar`. La recuperación y la comparación de contenido se hacen después del cronómetro del despliegue.


In [4]:
auditorias = [json.loads((carpeta / f"parte_{i}.json").read_text()) for i in range(2)]
display(pd.DataFrame(auditorias)[["particion", "filas_entrada", "filas_salida", "hilos_utilizados"]])
assert sum(a["filas_salida"] for a in auditorias) == 108733
assert sum(a["hilos_utilizados"] for a in auditorias) == 4
display(pd.DataFrame([json.loads((carpeta / "validacion.json").read_text())]))
display(pd.DataFrame([json.loads((carpeta / "union.json").read_text())]))

,particion,filas_entrada,filas_salida,hilos_utilizados
0,0,54118,54118,2
1,1,54615,54615,2


,filas,columnas,coincide_referencia_python,sha256_csv,auditoria_coincide_referencia
0,108733,42,True,7ac454c8bc4420dd7f4d36fd168820f1b2e5f608629d95...,True


,implementacion,filas_dataset,filas_entrada,duplicados_exactos_eliminados,duplicados_id_eliminados,edades_invalidas_convertidas_a_nulo,filas_salida,hilos_solicitados,hilos_utilizados,particion,particiones,lectura_segundos,preparacion_global_segundos,transformacion_filas_segundos,etl_segundos,escritura_segundos,total_segundos,openmp_version
0,C++17/merge,108733,108733,0,0,0,108733,1,1,0,2,0.177580,0,0,0,0.130141,0.307721,201511


## Resultado del avance 3

El Job indexado completó sus dos particiones, el Job de unión finalizó correctamente y la salida recuperada coincidió con la referencia. Se conservan Jobs, auditorías, registros de finalización y validaciones por repetición, además del estado completo de los Pods de la última corrida. El notebook 04 compara su tiempo de pared con las demás implementaciones.


## Fuentes técnicas

- [OpenMP: distribución de iteraciones](https://www.openmp.org/spec-html/5.2/openmpse66.html).
- [GCC: compilación con `-fopenmp`](https://gcc.gnu.org/projects/gomp/).
- [Kubernetes: Jobs paralelos e indexados](https://kubernetes.io/docs/concepts/workloads/controllers/job/).
- [Cornell: Ley de Amdahl](https://cvw.cac.cornell.edu/parallel/efficiency/amdahls-law).

Los resultados numéricos proceden exclusivamente de los CSV, auditorías y registros de esta entrega.
